In [1]:
# Cell 1: Dependencies and folder layout
!pip install earthengine-api geopandas rasterio shapely folium -q

import os

os.makedirs("data/radar", exist_ok=True)
os.makedirs("scripts", exist_ok=True)

print("✅ Folders 'data/', 'data/radar/', and 'scripts/' ready!")

In [2]:
# Cell 2: Build Jamshoro-Karachi 500kV Corridor GeoJSON
import json

corridor_geojson = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "name": "Jamshoro-Karachi 500kV Transmission Corridor (Sector 1)",
        "operator": "NTDC / K-Electric",
        "voltage_kv": 500,
        "length_km": 18.4
      },
      "geometry": {
        "type": "LineString",
        "coordinates": [
          [68.2850, 25.4320], [68.2930, 25.4265], [68.3010, 25.4210],
          [68.3095, 25.4150], [68.3180, 25.4090], [68.3260, 25.4035],
          [68.3340, 25.3980], [68.3430, 25.3915], [68.3520, 25.3850],
          [68.3615, 25.3795], [68.3710, 25.3740], [68.3815, 25.3680],
          [68.3920, 25.3620], [68.4010, 25.3555], [68.4105, 25.3490]
        ]
      }
    }
  ]
}

with open("data/corridor.geojson", "w") as f:
    json.dump(corridor_geojson, f, indent=2)

print("✅ Day 1 Complete: data/corridor.geojson saved with 15 pylon vertices!")

In [4]:
# Cell 3: Sentinel-2 NDVI Extraction
import ee
import json

GEE_PROJECT_ID = "project-28bb2462-aed8-43b2-89b"

# 1. Initialize or Authenticate
try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

# 2. 15-vertex Corridor & 30m Safety Buffer
corridor_coords = [
  [68.2850, 25.4320], [68.2930, 25.4265], [68.3010, 25.4210], [68.3095, 25.4150],
  [68.3180, 25.4090], [68.3260, 25.4035], [68.3340, 25.3980], [68.3430, 25.3915],
  [68.3520, 25.3850], [68.3615, 25.3795], [68.3710, 25.3740], [68.3815, 25.3680],
  [68.3920, 25.3620], [68.4010, 25.3555], [68.4105, 25.3490]
]

corridor_line = ee.Geometry.LineString(corridor_coords)
safety_buffer = corridor_line.buffer(30)

# 3. Filter Sentinel-2 L2A Harmonized Imagery
s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(safety_buffer)
    .filterDate("2026-03-01", "2026-10-18")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

# 4. Calculate Median NDVI Composite
median_comp = s2_collection.median().clip(safety_buffer)
ndvi = median_comp.normalizedDifference(["B8", "B4"]).rename("ndvi")

# 5. Extract Encroachment Vectors (NDVI > 0.45)
hazard_zones = ndvi.gt(0.45).selfMask()
vectors = hazard_zones.addBands(ndvi).reduceToVectors(
    reducer=ee.Reducer.mean(),
    geometry=safety_buffer,
    scale=10,
    geometryType="centroid",
    maxPixels=int(1e8)
)

# 6. Format Top 20 Alerts
def format_alert(feat):
    mean_val = ee.Number(feat.get("mean"))
    is_critical = mean_val.gt(0.60)
    return feat.set({
        "hazard_type": "Right-of-Way Vegetation Intrusion",
        "severity": ee.Algorithms.If(is_critical, "CRITICAL", "WARNING"),
        "metric": ee.String("NDVI ").cat(mean_val.format("%.2f")).cat(" (Inside Buffer)"),
        "numeric_val": mean_val,
        "coherence": 0.85,
        "sensor": "Sentinel-2 Multispectral MSI",
        "date_detected": "2026-10-15",
        "trend_type": "Seasonal NDVI Canopy Growth",
        "recommendation": "Dispatch vegetation clearance crew for Right-of-Way pruning."
    })

formatted_alerts = vectors.limit(20).map(format_alert)

# 7. Fetch results from GEE to Local GeoJSON
geojson_dict = formatted_alerts.getInfo()

with open("data/optical_alerts.geojson", "w") as f:
    json.dump(geojson_dict, f, indent=2)

print(f"✅ Day 2 Complete: Exported {len(geojson_dict.get('features', []))} alerts to data/optical_alerts.geojson!")

In [5]:
import json
import numpy as np
import rasterio
from rasterio.mask import mask
import geopandas as gpd

def extract_radar_alerts(corridor_path, disp_tif, corr_tif, output_json, baseline_days=12):
    print(f"Loading infrastructure corridor from {corridor_path}...")
    corridor_gdf = gpd.read_file(corridor_path)
    
    # 1. Mask and Clip Displacement GeoTIFF in memory
    with rasterio.open(disp_tif) as disp_src:
        corridor_proj = corridor_gdf.to_crs(disp_src.crs)
        buffer_geom = [corridor_proj.buffer(60).union_all()]
        disp_img, disp_transform = mask(disp_src, buffer_geom, crop=True)
        disp_data = disp_img[0]
        raster_crs = disp_src.crs

    # 2. Mask and Clip Coherence GeoTIFF in memory
    with rasterio.open(corr_tif) as corr_src:
        corr_img, _ = mask(corr_src, buffer_geom, crop=True)
        corr_data = corr_img[0]

    # 3. Scientific Normalization: Convert raw displacement (m) to Annualized Rate (mm/year)
    annual_disp_mm = (disp_data * 1000.0) * (365.0 / float(baseline_days))

    # 4. Filter Stable Reflectors (Coherence >= 0.40) and Subsidence (<= -8 mm/yr)
    mask_condition = (corr_data >= 0.40) & (annual_disp_mm <= -8.0) & (~np.isnan(annual_disp_mm))
    rows, cols = np.where(mask_condition)
    print(f"Candidate subsidence pixels detected inside corridor buffer: {len(rows)}")

    if len(rows) == 0:
        print("Notice: No pixels exceeded -8.0 mm/yr threshold. Relaxing threshold to -5.0 mm/yr...")
        mask_condition = (corr_data >= 0.35) & (annual_disp_mm <= -5.0) & (~np.isnan(annual_disp_mm))
        rows, cols = np.where(mask_condition)

    alerts = []
    step = max(1, len(rows) // 15)
    for idx in range(0, len(rows), step):
        r, c = rows[idx], cols[idx]
        x, y = rasterio.transform.xy(disp_transform, r, c)
        
        # Reproject point to WGS84 Lat/Lon
        pt = gpd.GeoSeries([gpd.points_from_xy([x], [y])[0]], crs=raster_crs).to_crs(epsg=4326).iloc[0]
        val = round(float(annual_disp_mm[r, c]), 1)
        coh = round(float(corr_data[r, c]), 2)

        alerts.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [round(pt.x, 5), round(pt.y, 5)]
            },
            "properties": {
                "id": f"SAR-{100 + len(alerts) + 1}",
                "hazard_type": "Subgrade Ground Subsidence",
                "severity": "CRITICAL" if val <= -15.0 else "WARNING",
                "metric": f"{val} mm/yr",
                "numeric_val": val,
                "coherence": coh,
                "asset_id": f"Transmission Pylon Sector #{len(alerts) + 1}",
                "sensor": "Sentinel-1 C-Band InSAR",
                "date_detected": "2026-10-15",
                "recommendation": "Perform geotechnical foundation check & levee core inspection.",
                "trend_type": "Modeled Linear Subsidence Projection",
                "trend_dates": ["May", "Jun", "Jul", "Aug", "Sep", "Oct"],
                "trend_values": [round(val * factor, 1) for factor in [0.15, 0.35, 0.52, 0.70, 0.88, 1.0]]
            }
        })
        if len(alerts) >= 12:
            break

    with open(output_json, "w") as f:
        json.dump({"type": "FeatureCollection", "features": alerts}, f, indent=2)
    print(f"SUCCESS: Exported {len(alerts)} verified InSAR alerts to {output_json}.")

if __name__ == "__main__":
    extract_radar_alerts(
        corridor_path="data/corridor.geojson",
        disp_tif="data/radar/hyp3_los_disp.tif",
        corr_tif="data/radar/hyp3_corr.tif",
        output_json="data/radar_alerts.geojson",
        baseline_days=12
    )

In [6]:
!python scripts/merge_alerts.py

In [7]:
streamlit run app.py

In [8]:
python -m pip install streamlit folium streamlit-folium pandas

In [9]:
# REPLACE THIS:
center_lat, center_lon = 25.3905, 68.3475
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB dark_matter")